# Étape 4 — Apprentissage Non Supervisé

Cas d'usage : **Segmentation des profils de trajets (K-Means)**

Objectif : identifier des profils types de courses (navetteur, touriste, noctambule, etc.)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

df = pd.read_parquet('../data/yellow_tripdata_features.parquet')
print(f'Shape : {df.shape}')

## 4.1 Sélection et standardisation des features

In [ ]:
CLUSTER_FEATURES = ['trip_distance', 'duree_course', 'heure_journee', 'total_amount', 'vitesse_moyenne']

df_cluster = df[CLUSTER_FEATURES].dropna()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_cluster)

## 4.2 Méthode du coude (Elbow Method)

In [ ]:
inertias = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K_range, inertias, marker='o')
plt.xlabel('Nombre de clusters k')
plt.ylabel('Inertie')
plt.title('Méthode du coude')
plt.savefig('../figures/04_elbow.png', dpi=150)
plt.show()

## 4.3 Application K-Means avec k optimal

In [ ]:
K_OPTIMAL = 4

km = KMeans(n_clusters=K_OPTIMAL, random_state=RANDOM_STATE, n_init=10)
df_cluster['cluster'] = km.fit_predict(X_scaled)

print(df_cluster.groupby('cluster')[CLUSTER_FEATURES].mean().round(2))

## 4.4 Visualisation PCA 2D

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(9, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1],
                      c=df_cluster['cluster'], cmap='tab10', alpha=0.3, s=5)
plt.colorbar(scatter, label='Cluster')
plt.title('Clusters visualisés par PCA (2 composantes)')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.savefig('../figures/04_clusters_pca.png', dpi=150)
plt.show()

## 4.5 Interprétation des clusters

> À compléter après avoir analysé les résultats. Exemple :
>
> - **Cluster 0** : Courtes distances, heure de pointe → *Navetteur*
> - **Cluster 1** : Longues distances, nuit → *Touriste/Aéroport*
> - **Cluster 2** : ...